# Bilstm+features (embedded ---- good)


In [2]:
# =========================================================
# 3-CLASS VEDIC ACCENT PREDICTION (WEIGHTED, TOKEN LEVEL)
# CHAR + MORPH FEATURES (MASKED PER ROW)
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, Concatenate, RepeatVector
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
import unicodedata


# -------------------------
# PARAMETERS
# -------------------------
MAX_LEN = 32
CHAR_EMB_DIM = 64
LSTM_UNITS = 128
DROPOUT_RATE = 0.3
BATCH_SIZE = 64
EPOCHS = 10
TEST_SIZE = 0.2

# -------------------------
# FEATURE SPACE
# -------------------------
FEATURE_COLS = [
    "docu::case",
    "docu::gender",
    "docu::number",
    "lemmata classic::lemma",
    "lemmata classic::lemma type",
    "forms::additional features noun",
    "docu::person",
    "docu::mood",
    "docu::tense",
    "docu::voice",
    "forms::additional features verb"
]

# Small different embedding dims per feature
FEATURE_EMB_DIMS = {
    "docu::case": 4,
    "docu::gender": 2,
    "docu::number": 2,
    "lemmata classic::lemma": 8,
    "lemmata classic::lemma type": 4,
    "forms::additional features noun": 4,
    "docu::person": 2,
    "docu::mood": 3,
    "docu::tense": 3,
    "docu::voice": 2,
    "forms::additional features verb": 4
}

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_excel("finalRoorkeeCorpus.xlsx")

words_accented = df["docu::form"].astype(str).tolist()
words_clean = df["docu::lubotskypada"].astype(str).tolist()

#print(len(words_accented))

# -------------------------
# EXTRACT LABELS
# -------------------------
def extract_labels(accented, clean):
    labels = []
    for c in accented:
        if unicodedata.combining(c):
            if c == "\u0301":
                labels[-1] = 1
            elif c == "\u0300":
                labels[-1] = 2
        else:
            labels.append(0)
    return labels

all_labels = []
for a, c in zip(words_accented, words_clean):
    lab = extract_labels(a, c)
    if len(lab) == len(c):
        all_labels.append(lab)
    else:
        all_labels.append([0]*len(c))

# -------------------------
# BUILD CHAR VOCAB
# -------------------------
chars = set("".join(words_clean))
char2idx = {c: i+1 for i, c in enumerate(sorted(chars))}
char2idx["PAD"] = 0
VOCAB_SIZE = len(char2idx)

#print(f'vocab = {char2idx}')
#print(f'vocab size = {VOCAB_SIZE}')

# -------------------------
# ENCODE CHAR INPUT
# -------------------------
X_char = [[char2idx[c] for c in w] for w in words_clean]
X_char = pad_sequences(X_char, maxlen=MAX_LEN, padding="post")

y = pad_sequences(all_labels, maxlen=MAX_LEN, padding="post")
y = np.expand_dims(y, -1)

# -------------------------
# ENCODE FEATURES (MASK NA → 0)
# -------------------------
feature_inputs = {}
feature_vocab_sizes = {}
encoded_features = []

for col in FEATURE_COLS:
    values = df[col].fillna("MASK").astype(str).tolist()

    vocab = {"MASK": 0}
    for v in sorted(set(values)):
        if v != "MASK":
            vocab[v] = len(vocab)

    encoded = [vocab[v] for v in values]

    feature_vocab_sizes[col] = len(vocab)
    feature_inputs[col] = np.array(encoded)
    encoded_features.append(feature_inputs[col])

# -------------------------
# SPLIT
# -------------------------
split_data = train_test_split(
    X_char, y, *encoded_features,
    test_size=TEST_SIZE, random_state=42
)

X_char_train = split_data[0]
X_char_val = split_data[1]
y_train = split_data[2]
y_val = split_data[3]

feature_train = split_data[4::2]
feature_val = split_data[5::2]

# -------------------------
# TOKEN-LEVEL CLASS WEIGHTS
# -------------------------
mask = X_char_train != 0
y_train_flat = y_train.squeeze()[mask]

classes = np.unique(y_train_flat)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_flat
)

class_weights = dict(zip(classes, class_weights))

weights_tensor = tf.constant(
    [class_weights.get(i, 1.0) for i in range(3)],
    dtype=tf.float32
)

def weighted_loss(y_true, y_pred):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    weights = tf.gather(weights_tensor, y_true)
    return tf.reduce_mean(loss * weights)

# -------------------------
# MODEL
# -------------------------


# CHAR INPUT
char_input = Input(shape=(MAX_LEN,), name="char_input")

# IMPORTANT: mask_zero=False
char_emb = Embedding(
    VOCAB_SIZE,
    CHAR_EMB_DIM,
    mask_zero=False
)(char_input)

# FEATURE INPUTS
feature_input_layers = []
feature_embs = []

for col in FEATURE_COLS:
    inp = Input(shape=(1,), name=col)

    emb = Embedding(
        input_dim=feature_vocab_sizes[col],
        output_dim=FEATURE_EMB_DIMS[col]
    )(inp)

    emb = tf.keras.layers.Flatten()(emb)

    feature_input_layers.append(inp)
    feature_embs.append(emb)

# CONCAT FEATURE VECTOR
feature_vector = Concatenate()(feature_embs)

# REPEAT ACROSS TIMESTEPS
feature_repeat = RepeatVector(MAX_LEN)(feature_vector)

# CONCAT CHAR + FEATURES
x = Concatenate()([char_emb, feature_repeat])

# 🔥 MANUAL MASKING LAYER (this replaces mask_zero=True logic)
x = tf.keras.layers.Masking(mask_value=0.0)(x)

# LSTM
x = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True))(x)
x = Dropout(DROPOUT_RATE)(x)

outputs = Dense(3, activation="softmax")(x)

model = Model(
    inputs=[char_input] + feature_input_layers,
    outputs=outputs
)

model.compile(
    optimizer="adam",
    loss=weighted_loss,
    metrics=["accuracy"]
)

model.summary()

# -------------------------
# TRAIN
# -------------------------
history = model.fit(
    [X_char_train] + feature_train,
    y_train,
    validation_data=([X_char_val] + feature_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# -------------------------
# EVALUATION
# -------------------------
y_pred = model.predict([X_char_val] + feature_val)
y_pred = np.argmax(y_pred, axis=-1)

y_true = y_val.squeeze()

mask = X_char_val != 0
y_true_flat = y_true[mask]
y_pred_flat = y_pred[mask]

print("\nClassification Report:\n")
print(classification_report(y_true_flat, y_pred_flat))

print("Macro F1:", f1_score(y_true_flat, y_pred_flat, average="macro"))



# =========================================================
# SAVE XLSX WITH PREDICTIONS (INDEX SAFE)
# =========================================================

# --- Re-split WITH indices ---
split_data = train_test_split(
    X_char, y, np.arange(len(X_char)), *encoded_features,
    test_size=TEST_SIZE,
    random_state=42
)

X_char_train = split_data[0]
X_char_val = split_data[1]
y_train = split_data[2]
y_val = split_data[3]
idx_train = split_data[4]
idx_val = split_data[5]

feature_train = split_data[6::2]
feature_val = split_data[7::2]

# --- Predict on FULL dataset ---
full_preds = model.predict([X_char] + encoded_features)
full_preds = np.argmax(full_preds, axis=-1)

# --- Reconstruct accented word ---
def reconstruct_word(clean_word, pred_labels):
    result = ""
    for ch, lab in zip(clean_word, pred_labels):
        result += ch
        if lab == 1:
            result += "\u0301"
        elif lab == 2:
            result += "\u0300"
    return result

predicted_words = [
    reconstruct_word(w, p)
    for w, p in zip(words_clean, full_preds)
]

# --- Create column (default NA for all rows) ---
prediction_column = ["NA"] * len(df)

# Fill ONLY validation rows
for i in idx_val:
    prediction_column[i] = predicted_words[i]

# --- Insert at beginning ---
df.insert(0, "prediction", prediction_column)

# --- Save ---
df.to_excel("1.xlsx", index=False)

print("Saved file: bilstm_char_morph_predictions.xlsx")

34206
vocab = {' ': 1, "'": 2, '(': 3, ')': 4, '*': 5, '-': 6, ':': 7, '[': 8, ']': 9, 'a': 10, 'b': 11, 'c': 12, 'd': 13, 'e': 14, 'g': 15, 'h': 16, 'i': 17, 'j': 18, 'k': 19, 'l': 20, 'm': 21, 'n': 22, 'o': 23, 'p': 24, 'r': 25, 's': 26, 't': 27, 'u': 28, 'v': 29, 'y': 30, '|': 31, '~': 32, 'Å': 33, 'ñ': 34, 'ā': 35, 'ī': 36, 'ś': 37, 'ū': 38, '̀': 39, '́': 40, '̐': 41, 'ḍ': 42, 'ḥ': 43, 'ḷ': 44, 'ṃ': 45, 'ṅ': 46, 'ṇ': 47, 'ṛ': 48, 'ṝ': 49, 'ṣ': 50, 'ṭ': 51, '‖': 52, 'PAD': 0}
vocab size = 53


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ docu::case          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::gender        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::number        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lemmata             │ (None, 1)         │          0 │ -                 │
│ classic::lemma      │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lemmata             │ (None, 1)         │          0 │ -                 │
│ classic::lemma type │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forms::additional   │ (None, 1)         │          0 │ -                 │
│ features noun       │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::person        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::mood          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::tense         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::voice         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forms::additional   │ (None, 1)         │          0 │ -                 │
│ features verb       │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_13        │ (None, 1, 4)      │         36 │ docu::case[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_14        │ (None, 1, 2)      │         10 │ docu::gender[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_15        │ (None, 1, 2)      │          8 │ docu::number[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_16        │ (None, 1, 8)      │     80,120 │ lemmata           │
│ (Embedding)         │                   │            │ classic::lemma[0… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 321,975 (1.23 MB)

 Trainable params: 321,975 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.9936 - loss: 0.0626

KeyboardInterrupt: 

In [ ]:
'''
# =========================================================
# ADD PREDICTED ACCENTED WORDS TO EXCEL
# =========================================================

import unicodedata

# --- helper: rebuild accented word from clean + predictions ---
def rebuild_accented(clean_word, pred_labels):
    result = ""
    for ch, label in zip(clean_word, pred_labels):
        result += ch
        if label == 1:
            result += "\u0301"  # acute
        elif label == 2:
            result += "\u0300"  # grave
    return result

# --- build full prediction list (aligned with original df) ---
all_preds_full = model.predict([X_char] + encoded_features)
all_preds_full = np.argmax(all_preds_full, axis=-1)

predicted_words = []


for word, preds in zip(words_clean, all_preds_full):
    preds_trimmed = preds[:len(word)]
    predicted_words.append(rebuild_accented(word, preds_trimmed))

# --- insert as first column ---
df.insert(0, "PREDICTED_ACCENT", predicted_words)

# --- save updated file ---
df.to_excel("Vedaweb_with_predictions.xlsx", index=False)

print("Updated file saved as: Vedaweb_with_predictions.xlsx")
'''

1069/1069 ━━━━━━━━━━━━━━━━━━━━ 31s 29ms/step
Updated file saved as: Vedaweb_with_predictions.xlsx


In [ ]:
'''
Classification Report: correct corpus 2 - (paper wala.)

              precision    recall  f1-score   support

           0       0.98      0.94      0.96     46684
           1       0.63      0.84      0.72      5589
           2       0.36      0.96      0.53       108

    accuracy                           0.93     52381
   macro avg       0.66      0.91      0.74     52381
weighted avg       0.94      0.93      0.93     52381

Macro F1: 0.7351267633138258
'''

'\nClassification Report: correct corpus 2 - (paper wala.)\n\n              precision    recall  f1-score   support\n\n           0       0.98      0.94      0.96     46684\n           1       0.63      0.84      0.72      5589\n           2       0.36      0.96      0.53       108\n\n    accuracy                           0.93     52381\n   macro avg       0.66      0.91      0.74     52381\nweighted avg       0.94      0.93      0.93     52381\n\nMacro F1: 0.7351267633138258\n'

#----------------------------------------------------------------------------

# Xmer with feats (no attention mask but padding mask) --- good result

In [4]:
# =========================================================
# 3-CLASS VEDIC ACCENT PREDICTION (WEIGHTED, TOKEN LEVEL)
# TRANSFORMER + CHAR + MORPH FEATURES
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Dense, Dropout,
    Concatenate, RepeatVector,
    LayerNormalization, MultiHeadAttention
)

from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
import unicodedata

# -------------------------
# PARAMETERS
# -------------------------
MAX_LEN = 32
CHAR_EMB_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_TRANSFORMER_BLOCKS = 2
DROPOUT_RATE = 0.3
BATCH_SIZE = 64
EPOCHS = 10
TEST_SIZE = 0.2

# -------------------------
# FEATURE SPACE
# -------------------------

FEATURE_COLS = [
    "docu::case","docu::gender","docu::number",
    "lemmata classic::lemma","lemmata classic::lemma type",
    "forms::additional features noun","docu::person",
    "docu::mood","docu::tense","docu::voice",
    "forms::additional features verb"
]

FEATURE_EMB_DIMS = {
    "docu::case":4,"docu::gender":2,"docu::number":2,
    "lemmata classic::lemma":8,"lemmata classic::lemma type":4,
    "forms::additional features noun":4,"docu::person":2,
    "docu::mood":3,"docu::tense":3,"docu::voice":2,
    "forms::additional features verb":4
}

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_excel("finalRoorkeeCorpus.xlsx")

words_accented = df["docu::form"].astype(str).tolist()
words_clean = df["docu::form_clean"].astype(str).tolist()

# -------------------------
# LABEL EXTRACTION
# -------------------------
def extract_labels(accented, clean):
    labels = []
    for c in accented:
        if unicodedata.combining(c):
            if c == "\u0301":
                labels[-1] = 1
            elif c == "\u0300":
                labels[-1] = 2
        else:
            labels.append(0)
    return labels

all_labels = []
for a, c in zip(words_accented, words_clean):
    lab = extract_labels(a, c)
    all_labels.append(lab if len(lab)==len(c) else [0]*len(c))

# -------------------------
# CHAR VOCAB
# -------------------------
chars = set("".join(words_clean))
char2idx = {c:i+1 for i,c in enumerate(sorted(chars))}
char2idx["PAD"]=0
VOCAB_SIZE=len(char2idx)

X_char=[[char2idx[c] for c in w] for w in words_clean]
X_char=pad_sequences(X_char,maxlen=MAX_LEN,padding="post")

y=pad_sequences(all_labels,maxlen=MAX_LEN,padding="post")
y=np.expand_dims(y,-1)

# -------------------------
# ENCODE FEATURES
# -------------------------
feature_vocab_sizes={}
encoded_features=[]

for col in FEATURE_COLS:
    values=df[col].fillna("MASK").astype(str).tolist()
    vocab={"MASK":0}
    for v in sorted(set(values)):
        if v!="MASK":
            vocab[v]=len(vocab)
    encoded=[vocab[v] for v in values]
    feature_vocab_sizes[col]=len(vocab)
    encoded_features.append(np.array(encoded))

# -------------------------
# SPLIT
# -------------------------
split_data=train_test_split(
    X_char,y,*encoded_features,
    test_size=TEST_SIZE,random_state=42
)

X_char_train=split_data[0]
X_char_val=split_data[1]
y_train=split_data[2]
y_val=split_data[3]

feature_train=split_data[4::2]
feature_val=split_data[5::2]

# -------------------------
# CLASS WEIGHTS
# -------------------------
mask=X_char_train!=0
y_train_flat=y_train.squeeze()[mask]
classes=np.unique(y_train_flat)

class_weights=compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_flat
)

class_weights=dict(zip(classes,class_weights))

weights_tensor=tf.constant(
    [class_weights.get(i,1.0) for i in range(3)],
    dtype=tf.float32
)

def weighted_loss(y_true,y_pred):
    y_true=tf.cast(tf.squeeze(y_true,-1),tf.int32)
    loss=tf.keras.losses.sparse_categorical_crossentropy(y_true,y_pred)
    weights=tf.gather(weights_tensor,y_true)
    return tf.reduce_mean(loss*weights)

# -------------------------
# POSITIONAL EMBEDDING
# -------------------------
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self,max_len,vocab_size,embed_dim):
        super().__init__()
        self.token_emb=Embedding(vocab_size,embed_dim)
        self.pos_emb=Embedding(max_len,embed_dim)
        self.max_len=max_len

    def call(self,x):
        positions=tf.range(start=0,limit=self.max_len,delta=1)
        return self.token_emb(x)+self.pos_emb(positions)

# -------------------------
# TRANSFORMER BLOCK
# -------------------------
def transformer_block(x):
    attn=MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=x.shape[-1]
    )(x,x)
    x=LayerNormalization(epsilon=1e-6)(x+attn)

    ffn=Dense(FF_DIM,activation="relu")(x)
    ffn=Dense(x.shape[-1])(ffn)

    return LayerNormalization(epsilon=1e-6)(x+ffn)

# -------------------------
# MODEL
# -------------------------
char_input=Input(shape=(MAX_LEN,),name="char_input")
char_emb=PositionalEmbedding(
    MAX_LEN,VOCAB_SIZE,CHAR_EMB_DIM
)(char_input)

# FEATURE INPUTS
feature_inputs=[]
feature_embs=[]

for col in FEATURE_COLS:
    inp=Input(shape=(1,),name=col)
    emb=Embedding(
        input_dim=feature_vocab_sizes[col],
        output_dim=FEATURE_EMB_DIMS[col]
    )(inp)
    emb=tf.keras.layers.Flatten()(emb)
    feature_inputs.append(inp)
    feature_embs.append(emb)

feature_vector=Concatenate()(feature_embs)
feature_repeat=RepeatVector(MAX_LEN)(feature_vector)

# CONCAT CHAR + FEATURES
x=Concatenate()([char_emb,feature_repeat])

# Transformer encoder layers
for _ in range(NUM_TRANSFORMER_BLOCKS):
    x=transformer_block(x)
    x=Dropout(DROPOUT_RATE)(x)

outputs=Dense(3,activation="softmax")(x)

model=Model(
    inputs=[char_input]+feature_inputs,
    outputs=outputs
)

model.compile(
    optimizer="adam",
    loss=weighted_loss,
    metrics=["accuracy"]
)

model.summary()

# -------------------------
# TRAIN
# -------------------------
history=model.fit(
    [X_char_train]+feature_train,
    y_train,
    validation_data=([X_char_val]+feature_val,y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


# -------------------------
# EVALUATION
# -------------------------
y_pred=model.predict([X_char_val]+feature_val)
y_pred=np.argmax(y_pred,axis=-1)

y_true=y_val.squeeze()

mask=X_char_val!=0
y_true_flat=y_true[mask]
y_pred_flat=y_pred[mask]

print("\nClassification Report:\n")
print(classification_report(y_true_flat,y_pred_flat))
print("Macro F1:",f1_score(y_true_flat,y_pred_flat,average="macro"))



# =========================================================
# SAVE XLSX WITH PREDICTED WORD COLUMN
# =========================================================

# Get predictions for FULL dataset
y_pred_full = model.predict([X_char] + encoded_features)
y_pred_full = np.argmax(y_pred_full, axis=-1)

# Identify which rows were in training set
# train_test_split preserves order, so we recreate index split
all_indices = np.arange(len(X_char))
train_indices, val_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=42
)

train_indices = set(train_indices)

# Function to reconstruct accented word
def reconstruct_word(clean_word, pred_labels):
    result = []
    for char, label in zip(clean_word, pred_labels):
        result.append(char)
        if label == 1:
            result.append("\u0301")  # acute
        elif label == 2:
            result.append("\u0300")  # grave
    return "".join(result)

# Build predicted column
predicted_words = []

for i in range(len(words_clean)):
    if i in train_indices:
        predicted_words.append("NA")
    else:
        pred_labels = y_pred_full[i][:len(words_clean[i])]
        predicted_words.append(
            reconstruct_word(words_clean[i], pred_labels)
        )

# Insert as FIRST column
df.insert(0, "predicted_word", predicted_words)

# Save new file
df.to_excel("pred_xmer_feats_80_20.xlsx", index=False)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ docu::case          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::gender        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::number        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lemmata             │ (None, 1)         │          0 │ -                 │
│ classic::lemma      │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lemmata             │ (None, 1)         │          0 │ -                 │
│ classic::lemma type │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forms::additional   │ (None, 1)         │          0 │ -                 │
│ features noun       │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::person        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::mood          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::tense         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ docu::voice         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forms::additional   │ (None, 1)         │          0 │ -                 │
│ features verb       │                   │            │                   │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_26        │ (None, 1, 4)      │         36 │ docu::case[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_27        │ (None, 1, 2)      │         10 │ docu::gender[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_28        │ (None, 1, 2)      │          8 │ docu::number[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_29        │ (None, 1, 8)      │     80,120 │ lemmata           │
│ (Embedding)         │                   │            │ classic::lemma[0… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 475,201 (1.81 MB)

 Trainable params: 475,201 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
 24/428 ━━━━━━━━━━━━━━━━━━━━ 1:08 170ms/step - accuracy: 0.7684 - loss: 0.5704

KeyboardInterrupt: 

In [ ]:
'''
#model.save("vedic_accent_transformer_model.keras")

Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.83      0.90     46684
           1       0.42      0.81      0.56      5589
           2       0.05      0.96      0.09       108

    accuracy                           0.83     52381
   macro avg       0.49      0.87      0.52     52381
weighted avg       0.92      0.83      0.86     52381

Macro F1: 0.5180627280675072
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 43s 40ms/step
'''